# 03 — Market, trajectory, availability (Phase 3)

Plan: `docs/superpowers/plans/2026-08-29-phase3-market-trajectory-availability.md`. Inputs are the
Phase 1 tables and the Phase 2 artifacts (`models/phase2_*.json`). Every fit is leave-future-out.

In [1]:
import json

import numpy as np
import pandas as pd

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 300)

from scout import config
from scout.data import reep, understat
from scout.data import transfermarkt as tm_loader
from scout.identity import build_team_lineage, load_overrides
from scout.panel import identity, stints, team_season

LEAGUE_TO_COMP = {league: comp for comp, league in config.BIG5.items()}
COMPS = list(config.BIG5) + list(config.FEEDERS)

## Step 1 — The market target and its sample

What `value_july` looks like (log scale) for players with a Phase 2 contribution row; coverage by
league-season; how value relates to contribution, age, minutes, league and club strength in the raw
data. Candidates: value at 1 July of the stats season vs at 1 July of the next season; club tier as
the club's expected points, its squad-value rank, or its Elo on 1 July.

In [2]:
# Phase 2 contribution rows (Understat ids) -> Transfermarkt ids -> stints with the two valuations
contrib = pd.DataFrame(json.load(open(config.MODELS / "phase2_contribution.json")))
tm_panel = tm_loader.load_player_club_seasons(COMPS, list(config.SEASONS))
tm_clubs = tm_panel[["club_id", "club_name", "competition_id"]].drop_duplicates()
us = understat.load("player_season")
us["competition_id"] = us.league.map(LEAGUE_TO_COMP)
lineage = build_team_lineage(tm_clubs, {"understat": us[["competition_id", "team"]].drop_duplicates().rename(columns={"team": "team_name"})}, load_overrides("teams"))
us_ids = identity.resolve_provider("understat", us, identity.transfermarkt_side(tm_panel), lineage, reep.load_people()).drop_duplicates("provider_id").set_index("provider_id").tm_player_id
contrib["tm_player_id"] = contrib.player_id.astype(int).astype(str).map(us_ids)

st = stints.build(COMPS, list(config.SEASONS))
st["tm_player_id"] = st.tm_player_id.astype(str)
season_value = st.sort_values("minutes", ascending=False).drop_duplicates(["tm_player_id", "season"])[["tm_player_id", "season", "club_id", "competition_id", "value_july", "value_age_days_july"]]
players = tm_loader.load_table("players")[["player_id", "date_of_birth"]]
players["tm_player_id"] = players.player_id.astype(str)

rows = contrib.dropna(subset=["tm_player_id"]).merge(season_value, on=["tm_player_id", "season"], how="left", suffixes=("", "_tm"))
rows = rows.merge(players[["tm_player_id", "date_of_birth"]], on="tm_player_id", how="left")
rows["age"] = rows.season + 1 - pd.to_datetime(rows.date_of_birth).dt.year
nxt = season_value.assign(season=season_value.season - 1)[["tm_player_id", "season", "value_july"]].rename(columns={"value_july": "value_next_july"})
rows = rows.merge(nxt, on=["tm_player_id", "season"], how="left")

print(len(rows), "contribution rows |", f"with a Transfermarkt id {rows.tm_player_id.notna().mean():.1%}", f"| with value_july {rows.value_july.notna().mean():.1%}", f"| with next July's value {rows.value_next_july.notna().mean():.1%}")
print("value coverage by league-season:")
print(rows.groupby(["competition_id", "season"]).value_july.apply(lambda c: c.notna().mean()).unstack("season").round(2).to_string())

21427 contribution rows | with a Transfermarkt id 100.0% | with value_july 99.0% | with next July's value 78.0%
value coverage by league-season:
season          2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025
competition_id                                                                        
ES1             0.98  0.98  0.97  0.99  0.99  1.00  0.99  0.99  0.99  0.99  0.99  1.00
FR1             0.97  0.95  0.97  0.98  0.96  0.97  0.97  0.99  0.99  0.99  0.98  0.98
GB1             0.99  0.99  0.99  0.99  0.99  0.97  0.99  1.00  1.00  0.99  1.00  1.00
IT1             1.00  1.00  1.00  1.00  1.00  0.99  1.00  1.00  1.00  1.00  1.00  1.00
L1              1.00  1.00  0.99  0.99  1.00  1.00  0.98  1.00  1.00  0.99  0.99  0.99


In [3]:
# The target on a log scale, by role and season
valued = rows.dropna(subset=["value_july"]).copy()
valued["log_value"] = np.log10(valued.value_july)

print("log10 value_july by role:")
print(valued.groupby("role").log_value.describe(percentiles=[0.1, 0.5, 0.9]).round(2)[["count", "mean", "std", "10%", "50%", "90%"]].to_string())
print("\nmedian value (M€) by season:", valued.groupby("season").value_july.median().div(1e6).round(2).to_dict())

log10 value_july by role:
       count  mean   std   10%   50%   90%
role                                      
CB    4503.0  6.65  0.56  5.90  6.65  7.40
CM    4991.0  6.75  0.57  6.00  6.78  7.48
FB    4031.0  6.60  0.54  5.90  6.60  7.30
ST    2919.0  6.83  0.56  6.18  6.85  7.58
W     4770.0  6.82  0.57  6.08  6.85  7.54

median value (M€) by season: {2014: 3.0, 2015: 3.0, 2016: 3.5, 2017: 4.0, 2018: 5.0, 2019: 8.0, 2020: 6.5, 2021: 7.0, 2022: 7.0, 2023: 7.0, 2024: 8.0, 2025: 8.0}


In [4]:
# Raw relationships: binned means of log value against each candidate feature
def binned(frame, col, bins, label):
    cut = pd.cut(frame[col], bins)
    table = frame.groupby(cut, observed=True).log_value.agg(["mean", "size"]).round(2)
    print(f"\n{label}:")
    print(table.T.to_string())


binned(valued, "point", [0, 0.05, 0.1, 0.2, 0.3, 0.45, 0.6, 0.8, 1.2, 3], "contribution (shrunk point, per 90)")
binned(valued, "age", [16, 20, 22, 24, 26, 28, 30, 32, 34, 45], "age")
binned(valued, "minutes", [600, 900, 1500, 2000, 2500, 3500], "minutes")
print("\nleague:", valued.groupby("competition_id").log_value.mean().round(2).to_dict())
print("role × contribution tercile:")
valued["contrib_tercile"] = valued.groupby("role").point.transform(lambda x: pd.qcut(x, 3, labels=["low", "mid", "high"]))
print(valued.pivot_table(index="role", columns="contrib_tercile", values="log_value", aggfunc="mean", observed=True).round(2).to_string())


contribution (shrunk point, per 90):
point  (0.0, 0.05]  (0.05, 0.1]  (0.1, 0.2]  (0.2, 0.3]  (0.3, 0.45]  (0.45, 0.6]  (0.6, 0.8]  (0.8, 1.2]
mean          6.45         6.63        6.68         6.8         6.78         6.93        7.17         7.7
size       2149.00      5077.00     4670.00      2707.0      3096.00      2430.00      948.00       114.0

age:
age   (16, 20]  (20, 22]  (22, 24]  (24, 26]  (26, 28]  (28, 30]  (30, 32]  (32, 34]  (34, 45]
mean      6.43      6.64      6.78      6.83      6.84      6.82       6.7      6.49      6.13
size    519.00   1858.00   3350.00   3920.00   3815.00   3156.00    2362.0   1356.00    870.00

minutes:
minutes  (600, 900]  (900, 1500]  (1500, 2000]  (2000, 2500]  (2500, 3500]
mean           6.62         6.69          6.74           6.8          6.87
size        4220.00      6480.00       4226.00        3250.0       3024.00

league: {'ES1': 6.68, 'FR1': 6.51, 'GB1': 7.06, 'IT1': 6.68, 'L1': 6.71}
role × contribution tercile:
contrib_tercile

In [5]:
# Club tier candidates: expected points (Understat), squad value rank (Transfermarkt), Elo on 1 July (ClubElo)
from scout.data import clubelo
from scout.panel import elo as elo_panel

ts = team_season.build()
ts["competition_id"] = ts.league.map(LEAGUE_TO_COMP)
team_club = us[["competition_id", "team", "team_id"]].drop_duplicates().merge(lineage[["competition_id", "team_name", "club_id"]].rename(columns={"team_name": "team"}), on=["competition_id", "team"])
ts = ts.merge(team_club[["competition_id", "team_id", "club_id"]], on=["competition_id", "team_id"])
club_strength = ts[["competition_id", "season", "club_id", "expected_points_for"]]

squad_value = season_value.groupby(["competition_id", "season", "club_id"]).value_july.sum().rename("squad_value").reset_index()
squad_value["squad_rank"] = squad_value.groupby(["competition_id", "season"]).squad_value.rank(ascending=False)

elo_names = elo_panel.club_elo_names(COMPS, list(config.SEASONS))
club_elo = []
for (comp, season, club_id), _ in squad_value.groupby(["competition_id", "season", "club_id"]):
    name = elo_names.get(club_id)
    if name is None:
        continue
    e = elo_panel.elo_on_dates(clubelo.fetch_club(name), pd.Series([f"{season}-07-01"]))[0]
    club_elo.append((comp, season, club_id, e))
club_elo = pd.DataFrame(club_elo, columns=["competition_id", "season", "club_id", "club_elo_july"])

tiers = club_strength.merge(squad_value, on=["competition_id", "season", "club_id"], how="outer").merge(club_elo, on=["competition_id", "season", "club_id"], how="outer")
v = valued.merge(tiers, on=["competition_id", "season", "club_id"], how="left")
print("club-tier coverage among valued rows:", {c: f"{v[c].notna().mean():.1%}" for c in ["expected_points_for", "squad_rank", "club_elo_july"]})

# which tier explains most of the residual after contribution and age (within role and season)?
import statsmodels.formula.api as smf

base = smf.ols("log_value ~ C(role) * (point + age + I(age**2)) + C(competition_id) + C(season)", data=v).fit()
v["resid"] = base.resid
print(f"\nbase model R² (contribution, age, role, league, season): {base.rsquared:.3f}")
for c in ["expected_points_for", "squad_rank", "club_elo_july"]:
    sub = v.dropna(subset=[c])
    r = np.corrcoef(sub.resid, sub[c])[0, 1]
    extra = smf.ols(f"resid ~ {c}", data=sub).fit().rsquared
    print(f"  {c}: r with residual {r:+.3f} | residual variance explained {extra:.3f} (n={len(sub)})")

club-tier coverage among valued rows: {'expected_points_for': '99.3%', 'squad_rank': '99.3%', 'club_elo_july': '99.3%'}



base model R² (contribution, age, role, league, season): 0.375
  expected_points_for: r with residual +nan | residual variance explained 0.225 (n=21074)
  squad_rank: r with residual +nan | residual variance explained 0.383 (n=21074)
  club_elo_july: r with residual +nan | residual variance explained 0.279 (n=21074)


### Step 1 note — target and sample, from the outputs above

**Sample.** 21,427 Phase 2 contribution rows, 100% with a Transfermarkt id, 99.0% with a
valuation at 1 July of the stats season and 78.0% with one at the following 1 July (the missing
fifth is mostly the current season, which has no "next July" yet); coverage ≥ 0.95 in every
league-season. Log10 value has sd ≈ 0.56 in every role (a factor of ~3.6); the raw relationships
are the expected ones — a rise of ~1.25 log10 across the contribution bins, an age curve peaking
at 24–30 and falling by 0.7 log10 after 34, a minutes gradient, and league premia (Premier League
7.06 vs Ligue 1 6.51 in log10 — about ×3.5).

**Target: log value at the 1 July *after* the stats season.** `value_july` is dated at the start
of the season, so it prices the *previous* season's profile; the backtest and the resale model
need the price the market puts on a season once it has happened, which is the next 1 July. The
season-of-stats value stays as a feature candidate (the market's prior).

**Club tier: the club's Elo on 1 July, not its squad-value rank.** After contribution, age, role,
league and season (R² 0.375), squad-value rank explains 38% of the remaining variance, Elo 28%,
expected points 22% — but the rank contains the player's own value and is the same market
judging itself, so it would leak the target into a feature. Elo is an independent measure of the
club's strength. Rejected: squad-value rank (circular), expected points (weaker and it only
exists for the Big 5 — the candidate pool needs a tier for feeder clubs too).